# 02. 딥러닝 시계열 계보 — RNN에서 PatchTST까지

> **Day 01 — 제조 시계열 AI (3/5)**
> 항공기 엔진의 잔여수명(RUL)을 예측하며, RNN → Attention → Transformer → PatchTST로
> 이어지는 계보를 직접 구현으로 따라간다.
>
> **핵심 메시지**: *"Transformer는 갑자기 나타나지 않았다. 순환을 버려야 했던 이유가 있었다."*
> **이 노트북의 역할**: NB03(이상탐지)·NB04(표현학습)에서 쓸 도구(Attention)를 손으로 만들어 둔다.

---

## 📖 이 노트북의 스토리라인

> **"Transformer는 갑자기 나타나지 않았습니다. 순환을 버려야 했던 이유가 있었습니다."**

각 화살표에는 **앞 모델이 못 한 일**이 적혀 있습니다. 그것이 다음 모델을 불렀습니다.

```
RNN ──→ LSTM ──→ Seq2Seq ──→ +Attention ──→ Transformer ──→ PatchTST
   기억이      게이트로도    쪽지 한 장에    원본을 다시      순환을 버려      점이 아니라
   흐려진다     부족하다      담기지 않는다    펴 본다         병렬로           패치로
```

| | |
|---|---|
| **쓰는 데이터** | NASA C-MAPSS FD001 (실데이터) — 터보팬 엔진 100대의 정상→고장 전체 궤적, 21센서 |
| **이 노트북의 역할** | 오후에 쓸 **도구를 손으로 만든다**. Attention을 직접 조립해야 NB03에서 내부를 열어 볼 수 있다 |
| **앞에서 이어받는 것** | NB01의 슬라이딩 윈도우 · 시간 분할 · train-only 스케일링 |
| **다음으로 넘기는 것** | Attention 행렬 → NB03의 **XAI 재료**. Patch 개념 → NB04의 마스킹 복원. 이 엔진 데이터 → NB04의 도메인 적응 |

> ⚠️ **이 노트북은 점심을 사이에 두고 둘로 나뉩니다.**
> 오전(§1~§7)은 이론과 부품 제작, 오후(§8~§11)는 조립과 실습입니다.
> 오후는 오전 없이도 단독 실행되도록 만들어져 있으니, 세션이 끊겨도 걱정하지 마십시오.

---

> **실습 안내**
> `"""따라하기"""` 가 적힌 셀은 강사와 함께 직접 실행합니다. 주석을 보고 코드를 채워 주세요.
> `"""직접구현"""` 이 적힌 셀은 여러분이 직접 채워 봅니다. 정답은 노트북 맨 아래에 있습니다.
> 나머지 셀은 실행 결과를 확인하며 따라오시면 됩니다.
> 막히는 부분은 손을 들어 주세요.

## 목차

**[이론부 — 오전]**
1. 문제 설정 — 항공기 엔진 RUL 예측
2. RNN/LSTM — 회의록을 한 줄씩 읽는 서기
3. LSTM 베이스라인 — 이후 모든 비교의 기준선
4. Seq2Seq의 병목 — 쪽지 한 장의 한계
5. Attention의 착상 — Self-Attention 5줄 구현
6. Transformer Encoder 조립
7. 시계열에 그대로 쓰면 생기는 문제

**[실습부 — 오후]**
8. 오전 복습과 재설정
9. PatchTST의 두 아이디어 — Patching · Channel Independence
10. PatchTST 스켈레톤 채워넣기와 학습
11. LSTM vs PatchTST — 계보를 숫자로 닫기

In [ ]:
# 공통 준비 — Colab / 로컬 양쪽에서 동작
import os, random, warnings
import numpy as np
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} | device: {DEVICE}")
except ImportError:
    DEVICE = "cpu"
    print("PyTorch 미설치 — 다음 셀에서 설치합니다.")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print(f"Colab 환경: {IN_COLAB}")

In [ ]:
# ══ 실습 자료 위치 — 강사가 배포한 주소로 이 한 줄만 맞추면 됩니다 ══════════
DATA_REPO = "https://raw.githubusercontent.com/leejiyoon52/ai-course/main/day1"

# Google Drive로 배포받았다면, 위 줄 대신 아래 두 줄의 주석을 푸십시오.
# from google.colab import drive; drive.mount("/content/drive")
# DATA_REPO = "file:///content/drive/MyDrive/ai-course/day1"
# ═══════════════════════════════════════════════════════════════════════

import os, shutil, urllib.request
os.environ["MFG_DATA_BASE"] = DATA_REPO          # loaders.py가 이 값을 읽습니다

for f in ["mfg_datagen.py", "loaders.py"]:
    if os.path.exists(f):
        continue
    src = f"{DATA_REPO}/modules/{f}"
    try:
        if src.startswith("file://"):
            shutil.copy(src[len("file://"):], f)
        else:
            urllib.request.urlretrieve(src, f)
        print(f"다운로드 완료: {f}")
    except Exception:
        print(f"⚠️ {f} 를 받지 못했습니다 — 왼쪽 파일 탭에 직접 업로드해 주세요")

import mfg_datagen
import loaders
print(f"실습 모듈 준비 완료 — 자료 위치: {DATA_REPO}")

---
## 1. 문제 설정 — 항공기 엔진 RUL 예측

**RUL(Remaining Useful Life, 잔여 유효 수명)** 은 설비가 고장까지 몇 사이클을
더 버틸 수 있는가입니다. *설비의 남은 체력 게이지*라고 생각하면 됩니다.

- 데이터: NASA **C-MAPSS** — 항공기 터보팬 엔진 시뮬레이터. 엔진 100대가
  정상에서 고장까지 가는 전체 궤적을 **21개 센서**로 기록했습니다.
- 왜 이 문제인가: RUL을 알면 **정비 시점을 계획**할 수 있습니다. "고장 나면 고친다"가
  "80사이클 남았으니 다음 주 계획 정비에 넣는다"로 바뀝니다 — NB01의 예지보전 그 자체입니다.
- 제조 현장의 같은 문제: CNC 공구 수명, 펌프·컴프레서 베어링 수명, 배터리 셀 용량 열화.

In [ ]:
# C-MAPSS FD001 로드 — 실데이터 우선, 실패 시 합성 폴백
"""따라하기"""


In [ ]:
# 엔진 한 대의 일생 — 열화가 센서에 어떻게 새겨지는가
"""따라하기"""


In [ ]:
# 엔진마다 수명이 얼마나 다른가 — 수명 분포
lifes = train_raw.groupby("unit")["cycle"].max()
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(lifes, bins=25)
ax.set_xlabel("engine life (cycles)"); ax.set_ylabel("count")
ax.set_title("Engine lifetime distribution")
plt.tight_layout(); plt.show()
print(f"수명 범위: {lifes.min()} ~ {lifes.max()}사이클 (평균 {lifes.mean():.0f})")
print("같은 기종인데 수명이 2배 넘게 차이 납니다 — 평균 수명 교체(TBM)가 낭비이거나 위험한 이유입니다.")

In [ ]:
# RUL 라벨 만들기 — 고장까지 남은 사이클, 상한 125로 클리핑
"""따라하기"""


---
## 2. RNN/LSTM — 회의록을 한 줄씩 읽는 서기

> RNN은 *회의록을 한 줄씩 읽으며 기억을 갱신하는 서기*입니다 — 앞부분은 점점 흐려집니다.

RNN(Recurrent Neural Network, 순환 신경망)은 시점 순서대로 hidden state를 갱신합니다.

- **장기 의존성 소실**: 수백 스텝 앞의 정보는 갱신을 거치며 씻겨 나갑니다.
- **순차 연산 병목**: t 시점 계산은 t-1이 끝나야 시작됩니다 — GPU가 놀게 됩니다.

LSTM(Long Short-Term Memory)은 게이트로 기억 소실을 완화한 개선판입니다.
완전한 해결이 아니라 **완화**라는 점이 이후 이야기의 출발점입니다.

In [ ]:
# 기억이 씻겨 나가는 속도 — 갱신 게이트가 0.9씩만 통과시켜도
import torch

steps = torch.arange(0, 300)
for keep in [0.99, 0.95, 0.90]:
    plt.plot(steps, keep ** steps, label=f"keep {keep}")
plt.axhline(0.01, color="red", ls="--", lw=0.8)
plt.title("How much of step-0 information survives after t steps")
plt.xlabel("steps later"); plt.ylabel("surviving fraction")
plt.legend(); plt.tight_layout(); plt.show()
print("갱신마다 90%를 남겨도 44스텝이면 1% 아래로 떨어집니다.")
print("300사이클 엔진 이력의 초반 징후는 마지막 hidden state에 거의 남지 않습니다.")

In [ ]:
# 입력 준비 ① — 추세가 뚜렷한 센서 8개 선택, train 통계로만 스케일링
"""따라하기"""


In [ ]:
# 입력 준비 ② — 유닛별 슬라이딩 윈도우 (NB01 make_windows의 다변량 버전)
"""따라하기"""


In [ ]:
# PyTorch DataLoader 준비
"""따라하기"""


---
## 3. LSTM 베이스라인 — 이후 모든 비교의 기준선

> 베이스라인은 *신입 작업자의 감(勘)*입니다 — 이걸 못 이기면 그 모델은 필요 없습니다.

오늘 오후 PatchTST가 이길 상대가 바로 이 숫자입니다. 기준선 없는 성능 주장은 공허합니다.

In [ ]:
# LSTM RUL 모델 정의 — 마지막 hidden state로 회귀
"""따라하기"""


In [ ]:
# LSTM 학습 — 5 epoch 데모
"""따라하기"""


In [ ]:
# 기준선 확정 — 평가 엔진에서 MAE·RMSE
"""따라하기"""


In [ ]:
# 예측 vs 실제 산점도 — 오차가 어디에 몰리는지 본다
fig, ax = plt.subplots(figsize=(4.5, 4.2))
ax.scatter(true_l, pred_l, s=4, alpha=0.3)
ax.plot([0, 125], [0, 125], "r--", lw=1)
ax.set_xlabel("actual RUL"); ax.set_ylabel("predicted RUL")
ax.set_title("LSTM baseline")
plt.tight_layout(); plt.show()
print("빨간 대각선이 완벽한 예측입니다. 대각선에서 세로로 떨어진 거리가 오차입니다.")

---
## 4. Seq2Seq의 병목 — 쪽지 한 장의 한계

> *두 시간 회의를 쪽지 한 장으로 요약해 넘기기* — 길수록 빠뜨립니다.

번역을 위해 태어난 Seq2Seq 구조는 인코더가 입력 전체를 **고정 길이 컨텍스트 벡터
하나**로 압축해 디코더에 넘깁니다.

```
[x1 x2 x3 ... x300]  →  인코더  →  [ 벡터 1개 ]  →  디코더  →  출력
                                   ↑ 여기서 정보가 뭉개진다
```

- 입력이 길어질수록 벡터 하나에 눌러 담다 잃는 정보가 커집니다.
- 300사이클 엔진 이력의 초반 이상 징후는 쪽지에 적히지 못하고 사라집니다.
- 이 병목의 해법으로 나온 것이 Attention입니다 — 구현은 다음 절에서 직접 합니다.

In [ ]:
# 쪽지 한 장 실험 — 서로 다른 두 이력을 벡터 하나로 압축하면 구별이 사라진다
t = np.linspace(0, 1, 300)
normal = 0.3 * np.sin(2 * np.pi * 12 * t) + t          # 정상 열화 이력
spiked = normal.copy(); spiked[20:24] += 2.5           # 초반에 이상 스파이크가 있던 이력

note1, note2 = normal.mean(), spiked.mean()            # '쪽지 한 장' = 벡터 1개로 압축
print(f"정상 이력 요약값   : {note1:.4f}")
print(f"스파이크 이력 요약값: {note2:.4f}")
print(f"차이               : {abs(note1 - note2):.4f} — 300스텝 중 4스텝의 사건은 요약에서 증발합니다")
print("압축하는 순간 '초반의 이상 징후'라는 정보는 복구할 수 없습니다. 그래서 원본을 다시 봐야 합니다.")

---
## 5. Attention의 착상 — 압축하지 말고 다시 보라

> Attention은 *요약에 의존하지 않고, 필요할 때마다 원본 일지를 다시 펴 보는 것*입니다.

**Scaled Dot-Product Attention** 수식은 한 줄입니다.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

- **Q(query)**: 지금 내가 찾고 싶은 것 / **K(key)**: 각 시점의 색인 / **V(value)**: 각 시점의 내용
- $QK^\top$: 모든 시점 쌍의 관련도 점수 / $\sqrt{d_k}$: 점수 폭주 방지 / softmax: 확률로 정규화
- 수식 그대로 코드 5줄입니다. 아래에서 직접 씁니다.

In [ ]:
# Self-Attention을 코드 5줄로 — 수식과 한 줄씩 대응시킨다
"""따라하기"""


In [ ]:
# √d_k 스케일링이 왜 필요한가 — 차원이 커지면 softmax가 한 점에 몰린다
for d in [8, 64, 512]:
    q, k = torch.randn(1, d), torch.randn(6, d)
    raw = (q @ k.T).squeeze()
    sm_raw = torch.softmax(raw, dim=-1)
    sm_scaled = torch.softmax(raw / d ** 0.5, dim=-1)
    print(f"d_k={d:4d} | 스케일 없음: max {sm_raw.max():.3f} | 스케일 적용: max {sm_scaled.max():.3f}")
print("\n스케일이 없으면 고차원에서 한 시점에 확률이 쏠려, 나머지 시점의 기울기가 죽습니다.")

In [ ]:
# Attention 행렬을 그림으로 — "누가 누구를 보는가"
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(A[0].detach().numpy(), cmap="viridis")
ax.set_xlabel("attends to (key)"); ax.set_ylabel("query")
ax.set_title("Attention weights (6x6)")
plt.colorbar(im)
plt.tight_layout()
plt.show()
print("행마다 합이 1인 확률 분포입니다. 각 시점이 어느 시점을 참조했는지가 그대로 보입니다.")
print("→ NB03에서는 바로 이 행렬이 '고장 원인 추적(XAI)'의 재료가 됩니다.")

---
## 6. Transformer Encoder 조립 — 순환을 버리고 병렬로

Attention만으로는 두 가지가 부족합니다.

1. **순서 정보가 없다** — Attention은 집합 연산이라 시점을 섞어도 결과가 같습니다.
   → **Positional Encoding(위치 부호화)** 을 더해 순서를 새깁니다.
2. **한 종류의 관계만 본다** — 열화 추세와 사이클 리듬은 다른 관점이 필요합니다.
   → **Multi-Head**: 여러 Attention을 병렬로 돌려 서로 다른 관계를 보게 합니다.

`nn.Transformer` 같은 완제품은 쓰지 않습니다. 부품을 직접 조립해야 NB03·NB04에서
내부를 열어 볼 수 있습니다.

In [ ]:
# Positional Encoding — 사인·코사인으로 위치를 새긴다
"""따라하기"""


In [ ]:
# Multi-Head Attention 직접 구현 — 여러 관점을 병렬로
"""따라하기"""


In [ ]:
# head마다 다른 관점 — 4개 head의 Attention 지도를 나란히
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for h, ax in enumerate(axes):
    ax.imshow(A[0, h].detach().numpy(), cmap="viridis")
    ax.set_title(f"head {h}")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Same input, four different relation maps")
plt.tight_layout(); plt.show()
print("학습 전이라 무늬는 무작위지만, 학습이 되면 head마다 추세·주기 등 다른 관계를 분담합니다.")

In [ ]:
# Encoder Block 조립 — Attention + FFN + 잔차 + 정규화
"""따라하기"""


---
## 7. 시계열에 그대로 쓰면 생기는 문제

Transformer를 시계열에 그대로 쓰면 두 벽에 부딪힙니다.

1. **점 하나는 단어 하나가 아니다** — 문장의 단어는 하나하나가 의미 단위지만,
   진동 센서의 0.1초 값 하나는 거의 무의미합니다. 의미는 **구간(파형 조각)** 에 있습니다.
2. **계산량 O(L²)** — Attention 행렬은 길이의 제곱으로 커집니다. 시계열은 쉽게 수천 스텝이 됩니다.

In [ ]:
# O(L²)을 몸으로 — 길이를 늘리며 Attention 행렬 크기와 시간을 잰다
import time

for L in [100, 400, 1600]:
    x = torch.randn(1, L, 32)
    t0 = time.time()
    _ = mha(x)
    dt = (time.time() - t0) * 1000
    print(f"L={L:5d} | Attention 행렬 {L}x{L} = {L * L:>9,}칸 | {dt:6.1f} ms")
print("\n길이 4배 → 행렬 16배. 1초 샘플링 하루치(86,400스텝)는 이대로면 불가능합니다.")
print("→ 오후의 PatchTST가 이 두 문제를 한 번에 다룹니다.")

---
### ☕ 여기까지 오전 — 점심 후 이어집니다

오전에 만든 것: **RUL 문제 정의, LSTM 기준선(MAE 기록), Self-Attention·Multi-Head·
Encoder Block 부품 일체.**
오후에는 이 부품으로 PatchTST를 조립해 기준선에 도전합니다. 식사 맛있게 하세요.

---
## 8. 오전 복습 — 우리는 왜 순환(recurrence)을 버렸는가

1. RNN 서기는 **긴 회의록의 앞부분을 잊는다** (장기 의존성 소실)
2. 한 줄씩 읽느라 **GPU가 논다** (순차 연산 병목)
3. Seq2Seq의 쪽지 한 장은 **긴 입력을 담지 못한다** (고정 벡터 병목)
4. Attention: **압축하지 말고 필요할 때 원본을 다시 보자** — 병렬 연산은 덤
5. 남은 숙제: 점 하나는 단어가 아니다 + O(L²) → **PatchTST**

```
RNN → LSTM → Seq2Seq → +Attention → Transformer → PatchTST
(기억)  (게이트)  (압축 병목)   (원본 참조)     (병렬)      (패치)
```

In [ ]:
# 실습부 재설정 — 이 셀부터 실행해도 오후 실습이 완주되도록 전부 다시 준비합니다
import os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (11, 3.5); plt.rcParams["axes.grid"] = True

import loaders
print(f"재설정 완료 | device: {DEVICE}")

In [ ]:
# 데이터 재준비 — 오전과 동일 (독립 실행 보장)
train_raw, _, _ = loaders.load_cmapss("FD001")
train = loaders.add_rul(train_raw, cap=125)
SENSORS = ["s2", "s3", "s4", "s7", "s11", "s12", "s15", "s21"]
WIN = 30

units = train["unit"].unique()
n_tr = int(len(units) * 0.8)
tr_units, te_units = units[:n_tr], units[n_tr:]
scaler = StandardScaler().fit(train.loc[train["unit"].isin(tr_units), SENSORS])
train[SENSORS] = scaler.transform(train[SENSORS])

def make_unit_windows(df, units, sensors, win=WIN, stride=2):
    X, y = [], []
    for u in units:
        g = df[df["unit"] == u]
        arr, rul = g[sensors].values, g["RUL"].values
        for i in range(0, len(g) - win, stride):
            X.append(arr[i : i + win]); y.append(rul[i + win - 1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_tr, y_tr = make_unit_windows(train, tr_units, SENSORS)
X_te, y_te = make_unit_windows(train, te_units, SENSORS, stride=1)
RUL_CAP = 125.0
dl_tr = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr / RUL_CAP)),
                   batch_size=256, shuffle=True)
dl_te = DataLoader(TensorDataset(torch.tensor(X_te), torch.tensor(y_te / RUL_CAP)),
                   batch_size=256, shuffle=False)
print(f"X_tr {X_tr.shape} | X_te {X_te.shape} (타깃은 0~1 정규화)")

---
## 9. PatchTST의 두 아이디어

**① Patching** — *파형을 초 단위가 아니라 사이클 단위로 끊어 읽기*

- 시계열을 점이 아니라 **패치(작은 구간)** 단위로 묶어 하나의 토큰으로 만듭니다.
- 점 하나는 무의미해도 패치 하나에는 파형의 의미가 담깁니다.
- 시퀀스 길이가 패치 수로 줄어들어 **O(L²) 계산량도 함께 준다**는 것이 묘수입니다.

$$L\ \text{시점} \;\xrightarrow{\;\text{길이 } P,\ \text{보폭 } S\;}\; N = \left\lfloor \frac{L - P}{S} \right\rfloor + 1\ \text{패치}$$

**② Channel Independence** — *온도계와 압력계를 같은 눈금자로 재지 않기*

- 센서(채널)마다 파형의 성격이 다릅니다. 억지로 섞어 넣는 대신 **채널별로 독립 처리**하고
  가중치만 공유합니다.
- 제조 다변량 센서(온도·압력·진동이 한 상에 오르는)에 직결되는 설계입니다.

In [ ]:
# Patch 분할 구현 — NB01의 슬라이딩 윈도우를 윈도우 '안'에 또 적용
"""따라하기"""


In [ ]:
# 패치가 파형을 어떻게 써는지 눈으로 확인
sig = X_tr[0, :, 0]
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(sig, lw=1, color="gray", label="window (s2)")
for j in range(0, len(sig) - PATCH + 1, STRIDE):
    ax.axvspan(j, j + PATCH, alpha=0.12)
ax.set_title(f"One window sliced into patches (P={PATCH}, S={STRIDE})")
ax.legend()
plt.tight_layout(); plt.show()
print("겹치며 미끄러지는 반투명 구간 하나하나가 '토큰'이 됩니다.")

In [ ]:
# P(패치 길이)와 S(보폭)를 바꾸면 토큰 수와 계산량이 어떻게 변하나
rows = []
for P_, S_ in [(5, 5), (10, 5), (10, 10), (15, 5)]:
    N_ = (WIN - P_) // S_ + 1
    rows.append({"P": P_, "S": S_, "tokens N": N_, "attn cells N²": N_ * N_,
                 "coverage": "overlap" if S_ < P_ else "no overlap"})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\n원래 길이 {WIN} → Attention {WIN * WIN}칸이던 것이 패치로 크게 줄었습니다.")
print("P는 '한 토큰에 담을 파형의 의미 단위' — 설비의 사이클 길이에 맞추는 것이 출발점입니다.")

---
## 10. PatchTST 스켈레톤 채워넣기와 학습

이제 두 부품을 **여러분이 직접** 만듭니다. 완성 모델은 아래에 따로 제공되므로,
막혀도 실습은 계속됩니다 — 대신 채운 뒤 shape 검증으로 스스로 확인합니다.

| 직접구현 | 만들 것 | 검증 |
|---|---|---|
| ① Patch 임베딩 | 패치(P차원) → 모델 차원(d_model) 투영 + 위치 부호화 | 출력 shape |
| ② Encoder 구성 | EncoderBlock을 n_layers만큼 쌓기 | 출력 shape |

In [ ]:
# 오전에 만든 부품 재정의 (실습부 독립 실행용 — 오전과 동일 코드)
def positional_encoding(L, d_model):
    pos = torch.arange(L).unsqueeze(1).float()
    i = torch.arange(0, d_model, 2).float()
    angle = pos / (10000 ** (i / d_model))
    pe = torch.zeros(L, d_model)
    pe[:, 0::2] = torch.sin(angle); pe[:, 1::2] = torch.cos(angle)
    return pe

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h, self.dk = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
    def forward(self, x):
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, L, self.h, self.dk).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)
        A = torch.softmax(q @ k.transpose(-2, -1) / self.dk ** 0.5, dim=-1)
        return self.out((A @ v).transpose(1, 2).reshape(B, L, D)), A

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
    def forward(self, x):
        a, A = self.attn(x)
        x = self.ln1(x + a)
        return self.ln2(x + self.ffn(x)), A

print("부품 준비 완료: positional_encoding / MultiHeadAttention / EncoderBlock")

In [ ]:
# Patch 임베딩부 — 패치를 d_model 차원 토큰으로 만든다
"""직접구현"""
def embed_patches(patches, proj, d_model=32):
    """patches: (B, C, N, P) → (B*C, N, d_model)  ※ 채널 독립: 채널을 배치로 접는다"""
    # TODO: 패치를 (B*C, N, P) 로 펴고 → proj 로 d_model 차원 투영
    #       positional_encoding(N, d_model) 을 더해 반환
    #       반환 shape: (B*C, N, d_model)
    raise NotImplementedError('직접 채워 보세요')


proj = nn.Linear(PATCH, 32)
tok = embed_patches(patches, proj)
print(f"패치 {tuple(patches.shape)} → 토큰 {tuple(tok.shape)}")
assert tok.shape == (B * C, N, 32), "shape이 다르면 다시 확인해 보세요"
print("통과 ✓ — 채널 8개가 각각 독립된 시퀀스로 인코더에 들어갈 준비가 되었습니다")


In [ ]:
# Encoder 구성부 — 블록을 n_layers만큼 쌓아 통과시킨다
"""직접구현"""
def build_encoder(d_model=32, n_heads=4, d_ff=64, n_layers=2):
    # TODO: EncoderBlock 을 n_layers 개 만들어 nn.ModuleList 로 반환
    raise NotImplementedError('직접 채워 보세요')


def encode(tokens, blocks):
    # TODO: blocks 를 차례로 통과시키며 tokens 를 갱신
    #       마지막 층의 Attention 도 함께 반환
    raise NotImplementedError('직접 채워 보세요')


blocks = build_encoder()
z, A_last = encode(tok, blocks)
print(f"인코딩 결과 {tuple(z.shape)} | 마지막 층 Attention {tuple(A_last.shape)}")
assert z.shape == tok.shape, "shape이 다르면 다시 확인해 보세요"
print("통과 ✓ — 이 두 연습이 그대로 아래 완성 모델의 내부입니다")


In [ ]:
# PatchTST 조립 — 완성 코드 (위 연습과 동일한 구조)
"""따라하기"""


In [ ]:
# PatchTST 학습 — LSTM과 같은 조건(5 epoch)
"""따라하기"""


In [ ]:
# PatchTST 평가 — 예측 vs 실제
"""따라하기"""


In [ ]:
# 평가 엔진 한 대의 수명 궤적 — 예측이 실제 열화를 따라가는가
u = te_units[0]
g = train[train["unit"] == u]
Xu, yu = make_unit_windows(train, [u], SENSORS, stride=1)
with torch.no_grad():
    pu = ptst(torch.tensor(Xu).to(DEVICE)).cpu().numpy() * RUL_CAP

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(yu, label="actual RUL", lw=2)
ax.plot(pu, label="PatchTST prediction", lw=1.2)
ax.set_xlabel("window index (time)"); ax.set_ylabel("RUL")
ax.set_title(f"Unit {u} — RUL trajectory")
ax.legend()
plt.tight_layout(); plt.show()
print("수명 말기(오른쪽)로 갈수록 예측이 실제에 붙는 것이 중요합니다 — 정비 판단이 걸린 구간이기 때문입니다.")

In [ ]:
# 학습된 Attention 지도 미리보기 — NB03 원인 추적의 복선
with torch.no_grad():
    xb1 = torch.tensor(X_te[:1]).to(DEVICE)
    B1, L1, C1 = xb1.shape
    p1 = xb1.transpose(1, 2).unfold(2, ptst.patch, ptst.stride)
    t1 = ptst.proj(p1.reshape(B1 * C1, ptst.n_patches, ptst.patch)) + ptst.pe
    for blk_ in ptst.blocks:
        t1, A1 = blk_(t1)

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(A1[0, 0].cpu().numpy(), cmap="viridis")
ax.set_xlabel("attends to (patch)"); ax.set_ylabel("query patch")
ax.set_title("Learned attention (s2 channel, head 0)")
plt.colorbar(im); plt.tight_layout(); plt.show()
print("학습된 모델이 어느 패치를 참조해 RUL을 읽는지가 보입니다 — NB03에서는 이것으로 원인 센서를 추적합니다.")

In [ ]:
# 오전을 건너뛰고 오후만 실행한 경우 대비 — LSTM 기준선이 없으면 여기서 다시 만든다
# (오전 셀을 실행했다면 이 셀은 재사용 메시지만 출력하고 지나갑니다)
if "mae_lstm" not in dir():
    class LSTMRul(nn.Module):
        def __init__(self, n_sensors, hidden=32):
            super().__init__()
            self.lstm = nn.LSTM(n_sensors, hidden, num_layers=1, batch_first=True)
            self.head = nn.Linear(hidden, 1)
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.head(out[:, -1]).squeeze(-1)
    lstm = LSTMRul(len(SENSORS)).to(DEVICE)
    n_params_lstm = sum(p.numel() for p in lstm.parameters())
    t_lstm = train_model(lstm, dl_tr)                      # 학습 약 40초 소요 (T4 기준)
    pred_l, true_l, mae_lstm, rmse_lstm = evaluate(lstm, dl_te)
print(f"LSTM 기준선 준비 완료 — MAE {mae_lstm:.2f} / RMSE {rmse_lstm:.2f}")

In [ ]:
# 수명 구간별 오차 — 판단이 걸린 말기 구간에서 누가 정확한가
bins = [(0, 30, "late (0-30)"), (30, 80, "mid (30-80)"), (80, 126, "early (80-125)")]
rows = []
for lo, hi, name in bins:
    m = (true_p >= lo) & (true_p < hi)
    rows.append({"RUL range": name,
                 "LSTM MAE": round(np.abs(pred_l[m] - true_l[m]).mean(), 2),
                 "PatchTST MAE": round(np.abs(pred_p[m] - true_p[m]).mean(), 2),
                 "n": int(m.sum())})
print(pd.DataFrame(rows).to_string(index=False))
print("\n평균 MAE가 같아도 말기(late) 정확도가 다르면 실무 가치는 완전히 다릅니다.")

In [ ]:
# 두 모델의 예측 산점도를 나란히
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, (p_, t_, name) in zip(axes, [(pred_l, true_l, "LSTM"), (pred_p, true_p, "PatchTST")]):
    ax.scatter(t_, p_, s=4, alpha=0.3)
    ax.plot([0, 125], [0, 125], "r--", lw=1)
    ax.set_xlabel("actual RUL"); ax.set_title(name)
axes[0].set_ylabel("predicted RUL")
plt.tight_layout(); plt.show()
print("대각선 주변에 더 촘촘히 붙는 쪽이 이긴 것입니다 — 다음 셀에서 숫자로 확정합니다.")

---
## 11. LSTM vs PatchTST — 계보를 숫자로 닫기

계보 이야기의 결론은 감상이 아니라 세 개의 숫자여야 합니다:
**성능, 학습 시간, 파라미터 수.**

In [ ]:
# 3축 비교표 — 오전 기준선과의 대결
"""따라하기"""


**표를 읽는 법 — "최고의 모델"은 없습니다**

- PatchTST의 강점은 **긴 문맥**(윈도우를 늘릴수록)과 **다변량 채널**에서 커집니다.
  30스텝의 짧은 윈도우에서는 LSTM과 격차가 작을 수 있습니다 — 그것이 정직한 결과입니다.
- LSTM은 여전히 가볍고, 짧은 시퀀스·엣지 장비 배포에서는 합리적 선택입니다.
- 선택 기준은 항상 **데이터 길이 × 채널 수 × 배포 환경**의 삼각형입니다.

> **현장 노트**
> 논문 성능표만 보고 모델을 고르면 안 되는 이유가 이 표에 있습니다.
> 논문 벤치마크는 수백 스텝 문맥·클린 데이터 기준입니다. 우리 라인의 윈도우 길이,
> 센서 수, 추론 장비 사양으로 **직접 재본 숫자**만이 선택의 근거가 됩니다.
> PoC 단계에서 이 비교표를 우리 데이터로 다시 만드는 것이 첫 번째 할 일입니다.

> **현장 노트**
> "학습 시간"도 성능입니다. 공정 조건이 바뀔 때마다 재학습해야 하는 모델이라면,
> 재학습 1회가 8시간인 모델과 40분인 모델은 운영상 완전히 다른 물건입니다.
> Concept Drift가 잦은 라인일수록 이 축의 가중치를 올려야 합니다.

---
## Self-check

### Q1. RNN이 긴 시계열에서 부딪히는 두 가지 근본 한계는 무엇입니까?

<details>
<summary>정답 보기</summary>

- **장기 의존성 소실**: hidden state를 순차 갱신하며 먼 과거 정보가 점점 씻겨 나갑니다.
- **순차 연산 병목**: t 시점은 t-1이 끝나야 계산할 수 있어 병렬화가 막힙니다.
- LSTM의 게이트는 소실을 완화할 뿐이며, 병목은 구조를 바꿔야(순환 제거) 풀립니다.

</details>

---

### Q2. Attention 수식에서 √d_k로 나누는 이유와 softmax의 역할은 무엇입니까?

<details>
<summary>정답 보기</summary>

- 차원이 커지면 내적 값이 커져 softmax가 한 점에 몰립니다(기울기 소실). √d_k가 이를 완화합니다.
- softmax는 관련도 점수를 **합이 1인 확률 분포**로 바꿔 "어디를 얼마나 볼지"의 가중치로 만듭니다.
- 그 가중치로 V를 가중합한 것이 문맥이 반영된 새 표현입니다.

</details>

---

### Q3. PatchTST의 Patching이 "의미"와 "계산량" 두 문제를 동시에 푸는 원리는 무엇입니까?

<details>
<summary>정답 보기</summary>

- 점 하나는 단어만큼의 의미가 없지만, **패치(파형 조각)** 에는 지역 패턴의 의미가 담깁니다.
- 토큰 수가 L개에서 N=⌊(L−P)/S⌋+1개로 줄어 Attention 계산량 O(L²)이 O(N²)로 감소합니다.
- 즉 토큰의 의미 밀도를 올리면서 시퀀스 길이를 줄이는, 한 수로 두 문제를 건드리는 설계입니다.

</details>

---

### Q4. [현장 판단] 센서 40개짜리 설비에 시계열 모델을 올리려 합니다. Channel Independence 관점에서 무엇을 먼저 검토해야 합니까?

<details>
<summary>정답 보기</summary>

- 채널 간 스케일·파형 성격이 제각각인지 확인합니다 — 그렇다면 채널 혼합 입력은 불리합니다.
- 채널 독립 처리(가중치 공유)는 채널 수가 늘어도 모델이 커지지 않아 40채널에 유리합니다.
- 단, 채널 간 상호작용(예: 압력↔온도 인과)이 핵심인 문제라면 독립 처리가 정보를 버릴 수 있어,
  상호작용을 별도 피처나 후단 모델로 보완할지 함께 판단해야 합니다.

</details>


---
## 다음 노트북 예고 — NB03. Anomaly Transformer와 원인 추적 (XAI)

오늘 손으로 만든 Attention 행렬, 사실 예측보다 더 값진 용도가 있습니다.

> **"왜 고장이라고 판단했는가"** — NB00에서 제시한 난제 ①.

다음 노트북은 실제 수처리장 펌프 5개월 기록으로 **문제의 성격**을 먼저 확인한 뒤,
원인 센서의 정답이 있는 압출기 데이터에서 **Attention 가중치를 열어
어느 센서가 원인인지 역추적**하고 그 지목이 맞았는지까지 채점합니다.
"정상은 멀리 있는 시점과도 대화하지만, 이상은 바로 옆하고만 대화한다" —
Anomaly Transformer의 이 착상이 탐지와 설명을 한 번에 해결합니다.

---
## 📎 부록 — `"""직접구현"""` 정답 코드

먼저 스스로 채워 본 뒤에 펼쳐 보시기 바랍니다.

<details>
<summary>Patch 임베딩부 — 패치를 d_model 차원 토큰으로 만든다</summary>

```python
# Patch 임베딩부 — 패치를 d_model 차원 토큰으로 만든다
"""직접구현"""
def embed_patches(patches, proj, d_model=32):
    """patches: (B, C, N, P) → (B*C, N, d_model)  ※ 채널 독립: 채널을 배치로 접는다"""
    B, C, N, P = patches.shape
    tokens = patches.reshape(B * C, N, P)        # 채널을 배치 차원으로 (Channel Independence)
    tokens = proj(tokens)                        # 선형 투영: P → d_model
    tokens = tokens + positional_encoding(N, d_model).to(tokens.device)  # 위치 부호화
    return tokens

proj = nn.Linear(PATCH, 32)
tok = embed_patches(patches, proj)
print(f"패치 {tuple(patches.shape)} → 토큰 {tuple(tok.shape)}")
assert tok.shape == (B * C, N, 32), "shape이 다르면 다시 확인해 보세요"
print("통과 ✓ — 채널 8개가 각각 독립된 시퀀스로 인코더에 들어갈 준비가 되었습니다")
```

</details>

<details>
<summary>Encoder 구성부 — 블록을 n_layers만큼 쌓아 통과시킨다</summary>

```python
# Encoder 구성부 — 블록을 n_layers만큼 쌓아 통과시킨다
"""직접구현"""
def build_encoder(d_model=32, n_heads=4, d_ff=64, n_layers=2):
    return nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])

def encode(tokens, blocks):
    A_last = None
    for blk in blocks:
        tokens, A_last = blk(tokens)             # 블록을 차례로 통과
    return tokens, A_last

blocks = build_encoder()
z, A_last = encode(tok, blocks)
print(f"인코딩 결과 {tuple(z.shape)} | 마지막 층 Attention {tuple(A_last.shape)}")
assert z.shape == tok.shape, "shape이 다르면 다시 확인해 보세요"
print("통과 ✓ — 이 두 연습이 그대로 아래 완성 모델의 내부입니다")
```

</details>
